1. Plotter for Client Script

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

CSV_FILENAME = Path.cwd() / "csv" / "robot_log_2026-11-23_23-44-59.csv"

SAVE_PLOTS = True
OUTPUT_FOLDER = "plots"

JOINT_COLUMNS = [
    "j1_angle",
    "j2_angle",
    "j3_angle",
    "j4_angle",
    "j5_angle",
    "j6_angle",
]


# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(CSV_FILENAME)

df = df.sort_values("simulation_time").reset_index(drop=True)

if SAVE_PLOTS:
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)


def save_plot(filename):
    if SAVE_PLOTS:
        plt.savefig(
            os.path.join(OUTPUT_FOLDER, filename),
            dpi=300,
            bbox_inches="tight",
        )


# ============================================================
# 1. JOINT ANGLES OVER TIME
# ============================================================

plt.figure(figsize=(12, 6))

for joint in JOINT_COLUMNS:
    plt.plot(
        df["simulation_time"],
        df[joint],
        linewidth=1.2,
        label=joint.replace("_", " ").upper(),
    )

plt.xlabel("Simulation time (s)")
plt.ylabel("Joint angle (degrees)")
plt.title("Robot Joint Angles")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=6)
plt.grid(alpha=0.3)
plt.tight_layout()

save_plot("joint_angles.png")
plt.show()


# ============================================================
# 2. INDIVIDUAL JOINT ANGLES
# ============================================================

fig, axes = plt.subplots(3, 2, figsize=(12, 9), sharex=True)

for ax, joint in zip(axes.flat, JOINT_COLUMNS):

    ax.plot(
        df["simulation_time"],
        df[joint],
        linewidth=1.2,
    )

    ax.set_title(joint.replace("_", " ").upper())
    ax.set_ylabel("Angle (degrees)")
    ax.grid(alpha=0.3)

for ax in axes[-1]:
    ax.set_xlabel("Simulation time (s)")

plt.tight_layout()

save_plot("individual_joints.png")
plt.show()


# ============================================================
# 3. JOINT ANGLE DISTRIBUTIONS
# ============================================================

fig, axes = plt.subplots(3, 2, figsize=(12, 9))

for ax, joint in zip(axes.flat, JOINT_COLUMNS):

    ax.hist(
        df[joint],
        bins=40,
        density=True,
        alpha=0.75,
        edgecolor="black",
    )

    ax.set_title(joint.replace("_", " ").upper())
    ax.set_xlabel("Joint angle (degrees)")
    ax.set_ylabel("Density")
    ax.grid(alpha=0.2)

plt.tight_layout()

save_plot("joint_distributions.png")
plt.show()


# ============================================================
# 4. JOINT ANGLES BY COMMAND
# ============================================================

commands = sorted(df["sequence"].dropna().unique())

fig, axes = plt.subplots(
    len(commands),
    1,
    figsize=(12, 3.5 * len(commands)),
    sharex=True,
    squeeze=False,
)

for ax, command in zip(axes.flat, commands):

    subset = df[df["sequence"] == command]

    for joint in JOINT_COLUMNS:

        ax.scatter(
            subset["simulation_time"],
            subset[joint],
            s=5,
            label=joint.replace("_", " ").upper(),
        )

    ax.set_title(f"Command {command}")
    ax.set_ylabel("Joint angle (degrees)")
    ax.grid(alpha=0.3)

axes[-1, 0].set_xlabel("Simulation time (s)")

handles, labels = axes[0, 0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="lower center",
    bbox_to_anchor=(0.5, 0),
    ncol=6,
)

plt.tight_layout(rect=(0, 0.05, 1, 1))

save_plot("joint_angles_by_command.png")
plt.show()


# ============================================================
# 5. MOTION SPEED OVER TIME
# ============================================================

plt.figure(figsize=(12, 4))

plt.plot(
    df["simulation_time"],
    df["motion_speed"],
    linewidth=1.2,
)

plt.xlabel("Simulation time (s)")
plt.ylabel("Motion speed")
plt.title("Robot Motion Speed")
plt.grid(alpha=0.3)
plt.tight_layout()

save_plot("motion_speed.png")
plt.show()


# ============================================================
# 6. JOINT ANGLE BOX PLOTS
# ============================================================

plt.figure(figsize=(10, 5))

plt.boxplot(
    [df[joint].dropna() for joint in JOINT_COLUMNS],
    tick_labels=[f"J{i}" for i in range(1, 7)],
    showfliers=True,
)

plt.xlabel("Joint")
plt.ylabel("Joint angle (degrees)")
plt.title("Joint Angle Distributions")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

save_plot("joint_boxplots.png")
plt.show()


# ============================================================
# 7. JOINT CORRELATION MATRIX
# ============================================================

correlation = df[JOINT_COLUMNS].corr()

fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(
    correlation,
    cmap="Greys",
    vmin=-1,
    vmax=1,
)

ax.set_xticks(range(6))
ax.set_yticks(range(6))

ax.set_xticklabels([f"J{i}" for i in range(1, 7)])
ax.set_yticklabels([f"J{i}" for i in range(1, 7)])

for i in range(6):
    for j in range(6):
        value = correlation.iloc[i, j]
        if pd.notna(value):
            ax.text(
                j,
                i,
                f"{value:.2f}",
                ha="center",
                va="center",
                color="white" if abs(value) > 0.6 else "black",
            )

plt.colorbar(im, ax=ax, label="Pearson correlation")
plt.title("Joint Angle Correlation")
plt.tight_layout()

save_plot("joint_correlation.png")
plt.show()


print(f"Loaded {len(df)} samples from {CSV_FILENAME}")
print(f"Recorded cycles: {df['iteration'].nunique()}")
print(f"Commands: {commands}")

if SAVE_PLOTS:
    print(f"Plots saved to: {OUTPUT_FOLDER}")

KeyError: 'simulation_time'

2. Plotter for .csv file from .gd run